# 57_fp16_batched_matmul

Audience: junior researcher

- Challenge path: `challenges/medium/57_fp16_batched_matmul`
- Source spec: [challenges/medium/57_fp16_batched_matmul/challenge.html](../challenge.html)
- Source implementation: [challenges/medium/57_fp16_batched_matmul/challenge.py](../challenge.py)


## Problem Statement

The original challenge HTML is embedded below so the notebook stays close to the repo source.

<p>
  Implement a batched matrix multiplication in FP16. Given a batch of matrices <code>A</code> of shape <code>[B, M, K]</code> and a batch of matrices <code>B</code> of shape <code>[B, K, N]</code>, compute the output batch <code>C</code> of shape <code>[B, M, N]</code> such that for each batch index <code>b</code>:
  \[
    C_b = A_b \times B_b
  \]
  All matrices are stored in row-major order and use 16-bit floating point numbers (FP16/<code>half</code>). Accumulation during multiplication should use FP32 for better precision before converting the final result to FP16.
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>External libraries are not permitted</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>Accumulation during multiplication should use FP32 for better precision before converting the final result to FP16</li>
  <li>The final result must be stored in the <code>C</code> array as <code>half</code></li>
</ul>

<h2>Example 1:</h2>
<pre>
Input:
B = 2, M = 2, K = 3, N = 2
A = [
  [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]],
  [[7.0, 8.0, 9.0], [10.0, 11.0, 12.0]]
]
B = [
  [[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]],
  [[6.0, 5.0], [4.0, 3.0], [2.0, 1.0]]
]
Output:
C = [
  [[22.0, 28.0], [49.0, 64.0]],
  [[92.0, 68.0], [128.0, 95.0]]
]
</pre>

<h2>Constraints</h2>
<ul>
  <li>1 &le; <code>B</code> &le; 128</li>
  <li>1 &le; <code>M</code>, <code>N</code>, <code>K</code> &le; 1024</li>

  <li>Performance is measured with <code>K</code> = 256, <code>M</code> = 256, <code>N</code> = 256</li>
</ul>


## Framework Coverage

This notebook collects the currently available solution artifacts for PyTorch, JAX, Triton, and MLX.

## Pytorch

Source: `challenges/medium/57_fp16_batched_matmul/solution/solution.pytorch.py`

In [ ]:
import torch


def solve(A: torch.Tensor, B: torch.Tensor, C: torch.Tensor, BATCH: int, M: int, N: int, K: int):
    A_f32 = A.view(BATCH, M, K).to(torch.float32)
    B_f32 = B.view(BATCH, K, N).to(torch.float32)
    C.copy_(torch.bmm(A_f32, B_f32).to(torch.float16))


## Jax

Source: `challenges/medium/57_fp16_batched_matmul/solution/solution.jax.py`

In [ ]:
import jax
import jax.numpy as jnp


@jax.jit
def solve(A: jax.Array, B: jax.Array, BATCH: int, M: int, N: int, K: int) -> jax.Array:
    A_f32 = A.reshape(BATCH, M, K).astype(jnp.float32)
    B_f32 = B.reshape(BATCH, K, N).astype(jnp.float32)
    return jnp.matmul(A_f32, B_f32).astype(jnp.float16)


## Triton

Source: `challenges/medium/57_fp16_batched_matmul/solution/solution.triton.py`

In [ ]:
import torch

try:
    import triton  # noqa: F401
    import triton.language as tl  # noqa: F401
except Exception:  # pragma: no cover
    triton = None
    tl = None


def solve(A: torch.Tensor, B: torch.Tensor, C: torch.Tensor, BATCH: int, M: int, N: int, K: int):
    A_f32 = A.view(BATCH, M, K).to(torch.float32)
    B_f32 = B.view(BATCH, K, N).to(torch.float32)
    C.copy_(torch.bmm(A_f32, B_f32).to(torch.float16))


## Mlx

Source: `challenges/medium/57_fp16_batched_matmul/solution/solution.mlx.py`

In [ ]:
def solve(A, B, BATCH: int, M: int, N: int, K: int):
    try:
        import mlx.core as mx

        A_f32 = mx.array(A).reshape(BATCH, M, K).astype(mx.float32)
        B_f32 = mx.array(B).reshape(BATCH, K, N).astype(mx.float32)
        return mx.matmul(A_f32, B_f32).astype(mx.float16)
    except Exception:
        import torch

        A_f32 = torch.as_tensor(A).reshape(BATCH, M, K).to(torch.float32)
        B_f32 = torch.as_tensor(B).reshape(BATCH, K, N).to(torch.float32)
        return torch.bmm(A_f32, B_f32).to(torch.float16)


## Verification Notes

Use `python scripts/verify_matrix_solutions.py` for the local matrix-operation verifier.
GPU-only Triton validation still depends on a remote NVIDIA environment.